## AUXILARY ##

In [ ]:
from PIL import Image
import numpy as np

def load_png(filename):
    # Open the PNG image file
    with Image.open(filename) as img:
        img = img.convert('RGBA')  # Ensure it has alpha channel
        # Extract info
        num_layers = 1  # PNGs are typically considered as a single-layer image
        num_channels = img.mode.count  # Count the number of channels based on the mode
        #get the dimensionality of the bits per channel through calculating the number of bits per channel
        bits_per_channel = 8
        # Convert image data to a numpy array
        array_n_channels = np.array(img)
    return (num_layers, num_channels, bits_per_channel, array_n_channels)

def load_bmp(filename):
    # Open the BMP image file
    with Image.open(filename) as img:
        img = img.convert('RGB')  # Ensure it is in RGB format
        # Extract info
        num_layers = 1  # BMPs are typically considered as a single-layer image
        num_channels = img.mode.count  # Count the number of channels based on the mode
        bits_per_channel = img.bits
        # Convert image data to a numpy array
        array_n_channels = np.array(img)
    return (num_layers, num_channels, bits_per_channel, array_n_channels)

from PIL import Image
import numpy as np

def export_rgba_png(filename, data, width, height):
    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, 4))
    elif data.ndim != 3 or data.shape != (height, width, 4):
        raise ValueError("Data must be a flat array or a 3D array with shape (height, width, 4).")
    
    # Create and save the image as PNG
    image = Image.fromarray(data, 'RGBA')
    image.save(filename, 'PNG', compress_level=0)

def export_rgba_bmp(filename, data, width, height):
    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, 4))
    elif data.ndim != 3 or data.shape != (height, width, 4):
        raise ValueError("Data must be a flat array or a 3D array with shape (height, width, 4).")

    # Create and save the image as BMP
    image = Image.fromarray(data, 'RGBA')
    image.save(filename, 'BMP')

    # Check if data is already in the correct shape (height, width, channels)
    if data.ndim == 1:
        data = data.reshape((height, width, -1))
    elif data.ndim != 3:
        raise ValueError("Data must be a flat array or a 3D array with appropriate channel information.")
    
    # Create an Image object from the numpy array
    # Assume that data shape includes a channel dimension, e.g., (H, W, C) where C can be 1 (L), 3 (RGB), or 4 (RGBA)
    if data.shape[2] == 1:
        mode = 'L'  # Grayscale
    elif data.shape[2] == 3:
        mode = 'RGB'
    elif data.shape[2] == 4:
        mode = 'RGBA'
    else:
        raise ValueError("Unsupported number of channels. Data must have 1, 3, or 4 channels.")

    # Create the image
    image = Image.fromarray(data.astype('uint8'), mode)

    # Save the image with no compression
    image.save(filename, 'PNG', compress_level=0)
import numpy as np
from pydub import AudioSegment
import pydub
# Function to load an MP3 file and convert it to a numpy array
def load_mp3_to_array(file_path):
    # Load the MP3 file
    audio = AudioSegment.from_mp3(file_path)
    
    # Convert to single-channel (mono) if not already
    if audio.channels > 1:
        audio = audio.set_channels(1)
    
    # Get the raw audio data as a byte string and convert to a numpy array
    samples = np.array(audio.get_array_of_samples())
    print(len(samples))
    return samples, audio.frame_rate
from PIL import Image
import numpy as np

MASK = 0b11111000

def analyze_data_loss(image_filename,MASK=0b11111000):
    # Open the image
    with Image.open(image_filename) as img:
        # Convert image to RGB if not already (to handle JPEG and grayscale BMP)
        img = img.convert('RGB')
        
        # Convert image data to a numpy array
        original_data = np.array(img)

    # Mask to zero out the last two bits (binary: 11111100)

    # Apply the mask
    modified_data = original_data & MASK

    # Compute the absolute difference
    difference = np.abs(original_data - modified_data)

    # Calculate average difference per pixel
    average_difference = np.mean(difference)

    # Return the average difference
    return average_difference

def mask_rgb_array(data,MASK=0b11111000):
    # Mask to zero out the last two bits (binary: 11111100)

    # Apply the mask to each R, G , B channel of each pixel
    modified_data = data & MASK
    
    return modified_data


# Example usage:
filename = 'exploring_mp3_solution/jg.png'
loss = analyze_data_loss(filename)
print(f"Average data loss per color channel per pixel: {loss}")

#truncate the last two bits of each color channel
num_layers, num_channels, bits_per_channel, array_n_channels = load_png(filename)

modified_data = mask_rgb_array(array_n_channels)

#show me the first pixel's data before and after truncation
print(array_n_channels[0][0])
print(modified_data[0][0])

export_rgba_png('jg_truncated.png', modified_data, array_n_channels.shape[1], array_n_channels.shape[0])

#load the truncated image, and subtract the truncated from the original's data
filename = 'jg_truncated.png'
truncated_data = load_png(filename)[3]

#find the highest loss
loss = np.max(np.abs(array_n_channels - truncated_data))
print(f"Highest data loss per color channel per pixel: {loss}")

## CHAPTER 1 ##

In [ ]:
import pydub
from pydub import AudioSegment
pad_audio_to_fit_image = True

def algorithm_1(image_filename,audio_filename):
    '''
        This algorithm will try to fit a 16 bit in the following way into a single pixel - R(11111VVV),G(11111VVV),B(11111VVV),(VVVVVVVV)
        Where the stored value into the pixel is taken by the index of the V, representing the Nth bit of the 16 bit value, and the rest of the bits are 0
        Or for 32 bit - R (11111VVV),G(11111VVV),B(11111VVV),A(11111VVV),(VVVVVVVV)
        Where the 32 bit value is devided by the original (11111),(11111),(11111) unmodified values of the rgb channels, and is written into the alpha channel
    '''
    
    #load data array, to be written to the image
    def bake_16bit_old_schema(V,R,G,B,A,type='subtract'):
        #check if V is 16 bit int
        if V > 65535:
            raise ValueError("V must be a 16 bit integer")
        #check if R,G,B are at least 3 bits away from the max value
        if R > 248 or G > 248 or B > 248:
            raise ValueError("R,G,B must be at least 3 bits away from the max value")
        
        #R(UUUUUV(0)V(1)V(2)), G(UUUUUV(3)V(4)V(5)), B(UUUUUV(6)V(7)V(8)), A(V(9)V(10)V(11)V(12)V(13)V(14)V(15)) 
        r = R | (V & 0b0000000000000111)
        g = G | ((V & 0b0000000000111000) >> 3)
        b = B | ((V & 0b0000000111000000) >> 6)
        #the alpha channel will contain the rest of the bits
        if type == 'subtract':
            a = 255 - (V & 0b1111111000000000) >> 9
        elif type == 'replace':
            a = (V & 0b1111111000000000) >> 9
        
        return (r,g,b,a)
    def bake_16bit(V, R, G, B, A, type='subtract'):
        # Check if V is a 16-bit int
        if V > 65535:
            raise ValueError("V must be a 16-bit integer")
        # Check if R, G, B are at least 3 bits away from the max value
        if R > 252 or G > 248 or B > 248:
            raise ValueError("R, G, B must be at least 3 bits away from the max value")
        
        # R(UUUUUUV(0)V(1)), G(UUUUUV(2)V(3)V(4)), B(UUUUUV(5)V(6)V(7)), A(V(8)V(9)V(10)V(11)V(12)V(13)V(14)V(15))
        r = R | (V & 0b0000000000000011)      # Red gets 2 bits
        g = G | ((V & 0b0000000000011100) >> 2)  # Green gets 3 bits
        b = B | ((V & 0b0000000011100000) >> 5)  # Blue gets 3 bits
        # The alpha channel will contain the rest of the bits
        if type == 'subtract':
            a = 255 - ((V & 0b1111111100000000) >> 8)
        elif type == 'replace':
            a = (V & 0b1111111100000000) >> 8
        
        return (r, g, b, a)

    def bake_32bit_old_schema(V,R,G,B,A,type='subtract'):
        #R(UUUUUV(0)V(1)V(2)), G(UUUUUV(3)V(4)V(5)), B(UUUUUV(6)V(7)V(8)), A(V(9)V(10)V(11)V(12)V(13)V(14)V(15)) 
        #the same structure here, but the 32bit value is devided by the avg of the unmodified R,G,B values, and the result,is dispersed into the R,G,B channels and A channel
        avg = (R+G+B)/3
        v_16 = V/avg
        R = R | (v_16 & 0b0000000000000111)
        G = G | ((v_16 & 0b0000000000111000) >> 3)
        B = B | ((v_16 & 0b0000000111000000) >> 6)
        #the alpha channel will contain the rest of the bits.
        if type == 'subtract':
            # A = 255 - (v_16 & 0b1111111000000000) >> 9
            A = 255 - (v_16 & 0b1111111000000000) >> 9
        elif type == 'replace':
            A = (v_16 & 0b1111111000000000) >> 9
        
        return (R,G,B,A)
    
    def bake_32bit(V, R, G, B, A,type='subtract'):
        # Check if V is within 32-bit range
        if not (0 <= V <= 0xFFFFFFFF):
            raise ValueError("V must be a 32-bit integer")

        # Ensure R, G, B are not too high to prevent overflow when adding bits
        if R > 252 or G > 248 or B > 248:
            raise ValueError("R, G, B must have space for 2, 3, and 3 bits respectively")

        # Calculate the average of the RGB values
        avg = (R + G + B) / 3

        # Scale down V from 32-bit to 16-bit by dividing by the average, then scaling to 16-bit range
        if avg == 0:
            raise ValueError("Average RGB cannot be zero")
        scaled_V = int((V / 0xFFFFFFFF) * 65535 / avg)

        # Embed the bits into the RGBA channels
        r = R | (scaled_V & 0b0000000000000011)               # 2 least significant bits to R
        g = G | ((scaled_V & 0b0000000000011100) >> 2)        # Next 3 bits to G
        b = B | ((scaled_V & 0b0000000011100000) >> 5)        # Next 3 bits to B
        a = (scaled_V & 0b1111111100000000) >> 8              # Top 8 bits to A

        return (r, g, b, a)
    
    #load the image, and extract the data
    num_layers, num_channels, bits_per_channel, array_n_channels = load_png(image_filename)
    
    #mask the data
    modified_data = mask_rgb_array(array_n_channels,MASK=0b11111000)
    
    #load the audio data
    audio_data, frame_rate = load_mp3_to_array(audio_filename)
    
    #check if the audio data is 16 or 32 bit
    if audio_data.max() > 65535:
        bake = bake_32bit
        tag = '32bit'
    else:
        bake = bake_16bit
        tag = '16bit'
    
    #check if the audio data is long enough to fit into the image
    
    if len(audio_data) < modified_data.size:
        #pad the audio data with silence, to fit into the image
        if pad_audio_to_fit_image == True:
            audio_data = np.pad(audio_data,(0,modified_data.size-len(audio_data)))
        else:        
            raise ValueError("Audio data is not long enough to fit into the image")
    
    #go by row and column, and embed the audio data into the image
    total = 0
    for i in range(modified_data.shape[0]):
        for j in range(modified_data.shape[1]):
            #get the audio data
            audio_value = audio_data[i*modified_data.shape[1]+j]
            #get the pixel data
            pixel_data = modified_data[i][j]
            #bake the audio data into the pixel data
            modified_data[i][j] = bake(audio_value,pixel_data[0],pixel_data[1],pixel_data[2],pixel_data[3])
            total = i*modified_data.shape[1]+j
    
    print(f"Total pixels modified: {total}")
    print(f"Total audio encoded into the image in seconds: {total/frame_rate}")
    
    #export the modified image
    export_rgba_png(f'encoded_image_{tag}.png', modified_data, array_n_channels.shape[1], array_n_channels.shape[0])

In [ ]:
from PIL import Image
import numpy as np
import wave

def decode_16bit_audio_from_image(image_filename, output_audio_filename):
    # Load the modified image
    with Image.open(image_filename) as img:
        data = np.array(img)

    # Prepare to collect the extracted audio samples
    audio_samples = []

    # Decode the data embedded in the image pixels
    for row in data:
        for pixel in row:
            r, g, b, a = pixel
            # Extract the embedded bits from each color channel and the alpha channel
            V_r = r & 0b00000011
            V_g = (g & 0b00000111) << 2
            V_b = (b & 0b00000111) << 5
            V_a = a << 8

            # Combine the bits to reconstruct the original 16-bit audio sample
            audio_sample = V_r | V_g | V_b | V_a
            audio_samples.append(int(audio_sample))  # Convert numpy int64 to Python int

    # Convert the audio samples to bytes
    audio_bytes = bytearray()
    for sample in audio_samples:
        audio_bytes += sample.to_bytes(2, byteorder='little', signed=False)

    # Write the audio bytes to a WAV file
    with wave.open(output_audio_filename, 'wb') as wav:
        wav.setnchannels(1)  # Mono audio
        wav.setsampwidth(2)  # 2 bytes per sample
        wav.setframerate(48000)  # Common sample rate
        wav.writeframes(audio_bytes)


In [ ]:
#jupyter visualisation of an image being opened and the audio being played
from IPython.display import display
from IPython.display import Audio
from PIL import Image
import numpy as np
import os
#visualise the image

image_to_encode = 'image_samples/8k_test_sample.png'
audio_to_encode = 'audio_samples/night_on_fire.mp3'

algorithm_1(image_to_encode,audio_to_encode)

In [ ]:
image_to_visualise_and_decode = 'encoded_image_16bit.png'
output_audio = 'extracted_audio_16bit.mp3'
display(Image.open(image_to_visualise_and_decode))
decode_16bit_audio_from_image(image_to_visualise_and_decode, output_audio)
print("Playing the extracted audio:")
display(Audio(output_audio))
os.system(f'aplay {output_audio}')


In [ ]:
import matplotlib.pyplot as plt

def draw_embedding_schema():
    # Create figure and axes
    fig, ax = plt.subplots(figsize=(10, 4))
    
    # Set up the diagram properties
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # Bit layout for 16-bit and 32-bit values
    bit_labels_16 = ['V0', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15']
    bit_labels_32 = ['V0', 'V1', ..., 'V31']  # Extending to 32 bits for simplicity

    # Define rectangles for RGBA channels (16-bit schema)
    channels = {
        'R': {'range': slice(0, 2), 'color': 'red'},
        'G': {'range': slice(2, 5), 'color': 'green'},
        'B': {'range': slice(5, 8), 'color': 'blue'},
        'A': {'range': slice(8, 16), 'color': 'magenta'}
    }

    # Plotting the bit distribution
    for i, bit in enumerate(bit_labels_16):
        x = i * 1.2  # X position for each bit
        for channel, props in channels.items():
            if i in range(*props['range'].indices(len(bit_labels_16))):
                rect = plt.Rectangle((x, 5), 1, 1, color=props['color'])
                ax.add_patch(rect)
                ax.text(x + 0.5, 5.5, bit, ha='center', va='center', color='white')
    
    # Adding channel labels
    ax.text(1, 4, 'R', ha='center', va='center', color='red', fontsize=12)
    ax.text(3.6, 4, 'G', ha='center', va='center', color='green', fontsize=12)
    ax.text(6.6, 4, 'B', ha='center', va='center', color='blue', fontsize=12)
    ax.text(12, 4, 'A', ha='center', va='center', color='magenta', fontsize=12)

    # Titles and annotations
    ax.text(10, 7, '16-bit Embedding Schema', ha='center', va='center', fontsize=16, color='black')

    plt.show()

draw_embedding_schema()


## CHAPTER 2 ##

In [ ]:
#alogirthm v2 - we define the masks for encoding and decoding, try to fiddle with the alpha bits, as they are masking the image.
RED_MASK = 0b00000011
GREEN_MASK = 0b00000111
BLUE_MASK = 0b00000111
ALPHA_MASK = 0b00000111

pad_audio_to_fit_image = True

#add masks as arguments for the bake functions
def bake_16_bit_v2(V,R,G,B,A,R_MASK=RED_MASK,G_MASK=GREEN_MASK,B_MASK=BLUE_MASK,A_MASK=ALPHA_MASK,type='subtract'):
    # Check if V is a 16-bit int
    if V > 65535:
        raise ValueError("V must be a 16-bit integer")
    # Check if R, G, B are at least 3 bits away from the max value
    if R > 252 or G > 248 or B > 248:
        raise ValueError("R, G, B must be at least 3 bits away from the max value")
    
    # Apply the masks to the RGB channels
    R = R | (V & R_MASK)
    G = G | ((V & G_MASK) >> 3)
    B = B | ((V & B_MASK) >> 6)
    # Apply the mask to the alpha channel
    if type == 'subtract':
        A = 255 - (V & A_MASK) >> 9
    elif type == 'replace':
        A = (V & A_MASK) >> 9
    
    return (R, G, B, A)

def bake_32_bit_v2(V,R,G,B,A,R_MASK=RED_MASK,G_MASK=GREEN_MASK,B_MASK=BLUE_MASK,A_MASK=ALPHA_MASK,type='subtract'):
    # Check if V is within 32-bit range
    if not (0 <= V <= 0xFFFFFFFF):
        raise ValueError("V must be a 32-bit integer")

    # Ensure R, G, B are not too high to prevent overflow when adding bits
    if R > 252 or G > 248 or B > 248:
        raise ValueError("R, G, B must have space for 2, 3, and 3 bits respectively")

    # Calculate the average of the RGB values
    avg = (R + G + B) / 3

    # Scale down V from 32-bit to 16-bit by dividing by the average, then scaling to 16-bit range
    if avg == 0:
        raise ValueError("Average RGB cannot be zero")
    scaled_V = int((V / 0xFFFFFFFF) * 65535 / avg)

    # Embed the bits into the RGBA channels
    R = R | (scaled_V & R_MASK)               # 2 least significant bits to R
    G = G | ((scaled_V & G_MASK) >> 3)        # Next 3 bits to G
    B = B | ((scaled_V & B_MASK) >> 6)        # Next 3 bits to B
    A = (scaled_V & A_MASK) >> 9              # Top 8 bits to A

    return (R, G, B, A)

def encode_16_bit_audio_into_image_v2(image_path,audio_path,output_path,R_MASK=RED_MASK,G_MASK=GREEN_MASK,B_MASK=BLUE_MASK,A_MASK=ALPHA_MASK):
    #load the image, and extract the data
    num_layers, num_channels, bits_per_channel, arra0b11111000y_n_channels = load_png(image_path)
    
    #mask the data
    modified_data = mask_rgb_array(array_n_channels,MASK=0b11111000)
    
    #load the audio data
    audio_data, frame_rate = load_mp3_to_array(audio_path)
    
    #check if the audio data is 16 or 32 bit
    if audio_data.max() > 65535:
        bake = bake_32_bit_v2
        tag = '32bit'
    else:
        bake = bake_16_bit_v2
        tag = '16bit'
    
    #check if the audio data is long enough to fit into the image
    
    if len(audio_data) < modified_data.size:
        #pad the audio data with silence, to fit into the image
        if pad_audio_to_fit_image == True:
            audio_data = np.pad(audio_data,(0,modified_data.size-len(audio_data)))
        else:        
            raise ValueError("Audio data is not long enough to fit into the image")
    
    #go by row and column, and embed the audio data into the image
    total = 0
    for i in range(modified_data.shape[0]):
        for j in range(modified_data.shape[1]):
            #get the audio data
            audio_value = audio_data[i*modified_data.shape[1]+j]
            #get the pixel data
            pixel_data = modified_data[i][j]
            #bake the audio data into the pixel data
            modified_data[i][j] = bake(audio_value,pixel_data[0],pixel_data[1],pixel_data[2],pixel_data[3],R_MASK,G_MASK,B_MASK,A_MASK)
            total = i*modified_data.shape[1]+j
    
    print(f"Total pixels modified: {total}")
    print(f"Total audio encoded into the image in seconds: {total/frame_rate}")
    
    #export the modified image
    export_rgba_png(output_path, modified_data, array_n_channels.shape[1], array_n_channels.shape[0])
    return output_path

def decode_16_bit_audio_from_image_v2(image_path, output_audio_path,R_MASK=RED_MASK,G_MASK=GREEN_MASK,B_MASK=BLUE_MASK,A_MASK=ALPHA_MASK):
    # Load the modified image
    with Image.open(image_path) as img:
        data = np.array(img)

    # Prepare to collect the extracted audio samples
    audio_samples = []

    # Decode the data embedded in the image pixels
    for row in data:
        for pixel in row:
            r, g, b, a = pixel
            # Extract the embedded bits from each color channel and the alpha channel
            V_r = r & R_MASK
            V_g = (g & G_MASK) << 3
            V_b = (b & B_MASK) << 6
            V_a = a << 9

            # Combine the bits to reconstruct the original 16-bit audio sample
            audio_sample = V_r | V_g | V_b | V_a
            audio_samples.append(int(audio_sample))  # Convert numpy int64 to Python int

    # Convert the audio samples to bytes
    audio_bytes = bytearray()
    for sample in audio_samples:
        audio_bytes += sample.to_bytes(2, byteorder='little', signed=False)

    # Write the audio bytes to a WAV file
    with wave.open(output_audio_path, 'wb') as wav:
        wav.setnchannels(1)  # Mono audio
        wav.setsampwidth(2)  # 2 bytes per sample
        wav.setframerate(48000)  # Common sample rate
        wav.writeframes(audio_bytes)
    
    return output_audio_path

In [ ]:
#encode the image
filename = 'image_samples/jg.png'
audio_filename = 'audio_samples/night_on_fire.mp3'
output_filename = 'encoded_image_v2.png'

encode_16_bit_audio_into_image_v2(filename,audio_filename,output_filename)
